## 🛡️ Implementando o "Data Firewall": Validação Automatizada com SQL

Nesta etapa, deixamos de lado a verificação visual ("olhar linha a linha") e assumimos o papel de Engenheiros de Dados. Vamos construir regras de validação que funcionam como um **Data Firewall**.

### O Conceito: Data Firewall
Assim como um firewall de rede bloqueia tráfego malicioso, um *Data Firewall* é uma camada de código que impede que dados de baixa qualidade avancem para as camadas de consumo (Dashboards de BI ou Modelos de Machine Learning). Se o dado não passar nas regras, ele é barrado ou alertado antes de gerar prejuízo.

### A Ferramenta: DuckDB revolucionando a análise local
Para executar nossas validações via SQL sem a complexidade de instalar um servidor de banco de dados (como PostgreSQL ou SQL Server), utilizaremos o **DuckDB**.

> **O que é o DuckDB?**
> É um sistema de gerenciamento de banco de dados SQL **OLAP (Online Analytical Processing)** e **em processo**.
> * **Sem Servidor:** Roda diretamente dentro do processo Python (como o SQLite), mas é otimizado para análise de dados massivos.
> * **Integração Nativa:** Sua grande vantagem é a capacidade de executar queries SQL complexas **diretamente em DataFrames do Pandas**, sem necessidade de exportar ou carregar dados.

#### Padronização e Segurança de Tipos (Casting)

In [1]:
import duckdb
import pandas as pd

In [2]:
df_vendas = pd.read_csv('./data/sales.csv', index_col=None)

In [3]:
# Inicializa conexão em memória
con = duckdb.connect(database=':memory:')

# Registra o DataFrame do Pandas como uma tabela virtual no DuckDB
con.register('tb_vendas_raw', df_vendas)

In [4]:
# QUERY 1: A Camada de "Sanitização" (Bronze -> Silver)
# Objetivo: Tentar converter os tipos. O que não for conversível vira NULL.
sql_sanitizacao = """
    CREATE OR REPLACE TABLE tb_vendas_typed AS
    SELECT
        transaction_id,
        customer_id,
        customer_name,
        -- Tenta converter data. Se for "31/02", vira NULL (Inválido) 2025/02/09
        TRY_CAST(transaction_date AS DATE) as transaction_date_clean,
        
        -- Tenta converter preço e quantidade para números
        TRY_CAST(unit_price AS DECIMAL(10,2)) as unit_price_clean,
        TRY_CAST(quantity AS INTEGER) as quantity_clean,
        TRY_CAST(total_amount AS DECIMAL(10,2)) as total_amount_clean,
        
        category,
        status,
        
        -- Mantém os dados originais para auditoria se necessário
        transaction_date as _raw_date,
        quantity as _raw_qty
    FROM tb_vendas_raw;
"""

con.execute(sql_sanitizacao)
print("Tabela 'tb_vendas_typed' criada com sucesso (Tipagem Segura aplicada).")

# Diferença entre o dado bruto e o limpo (onde falhou o cast)
df_check_types = con.execute("""
    SELECT _raw_qty, quantity_clean 
    FROM tb_vendas_typed 
    WHERE _raw_qty != CAST(quantity_clean AS VARCHAR) 
       OR quantity_clean IS NULL
    LIMIT 5
""").df()

display(df_check_types)

Tabela 'tb_vendas_typed' criada com sucesso (Tipagem Segura aplicada).


,_raw_qty,quantity_clean
0,dois,<NA>
1,10 un.,<NA>


#### O Motor de Regras de Qualidade (The Quality Engine)

In [5]:
# --- QUERY 2 CORRIGIDA E BLINDADA ---

# 1. Primeiro, vamos apagar a tabela antiga para ter certeza que não é "fantasma"
con.execute("DROP TABLE IF EXISTS tb_data_quality_report")

# 2. Recriamos a tabela com DATA FIXA (Hardcoded)
# Substituímos 'CURRENT_DATE()' por "DATE '2026-02-07'" na linha do erro_data_futura

sql_auditoria_fixa = """
CREATE OR REPLACE TABLE tb_data_quality_report AS
WITH quality_checks AS (
    SELECT
        transaction_id,
        
        -- Regras padrão
        CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END AS erro_null_customer,
        CASE WHEN transaction_date_clean IS NULL THEN 1 ELSE 0 END AS erro_null_date,
        CASE WHEN COUNT(*) OVER(PARTITION BY transaction_id) > 1 THEN 1 ELSE 0 END AS erro_duplicidade,
        
        CASE 
            WHEN abs(total_amount_clean - (unit_price_clean * quantity_clean)) > 0.01 
            THEN 1 ELSE 0 
        END AS erro_calculo_matematico,
        
        CASE WHEN unit_price_clean < 0 THEN 1 ELSE 0 END AS erro_preco_negativo,
        
        -- --- A MUDANÇA ESTÁ AQUI EMBAIXO ---
        -- Forçamos a regra: Só é erro se for maior que 07/Fev/2026
        CASE 
            WHEN transaction_date_clean > DATE '2026-02-07' THEN 1 
            ELSE 0 
        END AS erro_data_futura,
        -- -----------------------------------
        
        CASE 
            WHEN category NOT IN ('Eletrônicos', 'Livros', 'Casa', 'Moda', 'Brinquedos') 
            THEN 1 ELSE 0 
        END AS erro_categoria_invalida

    FROM tb_vendas_typed
)
SELECT 
    *,
    (erro_null_customer + erro_null_date + erro_duplicidade + 
     erro_calculo_matematico + erro_preco_negativo + 
     erro_data_futura + erro_categoria_invalida) AS total_erros
FROM quality_checks;
"""

# 3. Executamos a Query (ESSA LINHA É A MAIS IMPORTANTE)
con.execute(sql_auditoria_fixa)
print("✅ Tabela de Auditoria reconstruída com data de corte fixa em 07/02/2026.")

# --- VERIFICAÇÃO IMEDIATA ---
# Vamos checar agora se sobrou alguém com erro de data futura que seja de 2025
check = con.execute("""
    SELECT count(*) as erros_restantes
    FROM tb_data_quality_report
    WHERE erro_data_futura = 1
""").df()

print("\nSe o número abaixo for 0 (ou próximo disso), o problema foi resolvido:")
display(check)

✅ Tabela de Auditoria reconstruída com data de corte fixa em 07/02/2026.

Se o número abaixo for 0 (ou próximo disso), o problema foi resolvido:


,erros_restantes
0,6


#### O Painel de Controle (Dashboard de Qualidade)

In [6]:
# QUERY 3: Resumo Executivo de Qualidade (Scorecard)

sql_scorecard = """
    SELECT 
        COUNT(*) as total_transacoes,
        
        -- Métricas de Falha (Contagem Absoluta)
        SUM(erro_null_customer) as falhas_cliente_nulo,
        SUM(erro_duplicidade) as falhas_duplicidade,
        SUM(erro_calculo_matematico) as falhas_matematica,
        SUM(erro_preco_negativo) as falhas_preco_negativo,
        SUM(erro_categoria_invalida) as falhas_padronizacao,
        
        -- Métrica de Qualidade (% de linhas perfeitas)
        ROUND(
            (SUM(CASE WHEN total_erros = 0 THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 
        2) as data_quality_score_percent
        
    FROM tb_data_quality_report;
"""

df_scorecard = con.execute(sql_scorecard).df()

print("--- Data Quality Scorecard (Resumo) ---")
display(df_scorecard.T) # Transpose para facilitar leitura vertical

--- Data Quality Scorecard (Resumo) ---


,0
total_transacoes,2050.0
falhas_cliente_nulo,107.0
falhas_duplicidade,100.0
falhas_matematica,72.0
falhas_preco_negativo,10.0
falhas_padronizacao,169.0
data_quality_score_percent,79.8


#### Investigando os "Culpados" (Drill-down)

In [7]:
# QUERY 4: Isolando as transações com Erro de Cálculo Financeiro (Consistência)

sql_investigacao = """
    SELECT 
        t.transaction_id,
        t.unit_price_clean,
        t.quantity_clean,
        t.total_amount_clean,
        (t.unit_price_clean * t.quantity_clean) as valor_esperado,
        'ERRO DE LÓGICA' as diagnostico
    FROM tb_vendas_typed t
    JOIN tb_data_quality_report r ON t.transaction_id = r.transaction_id
    WHERE r.erro_calculo_matematico = 1
    ORDER BY t.transaction_id
    LIMIT 10;
"""

print("--- Transações com Inconsistência Financeira (Audit Trail) ---")
df_investigacao = con.execute(sql_investigacao).df()
display(df_investigacao)

--- Transações com Inconsistência Financeira (Audit Trail) ---


,transaction_id,unit_price_clean,quantity_clean,total_amount_clean,valor_esperado,diagnostico
0,0400774f-34c7-4fd5-ab78-4166e59f20e7,876.95,2,438.48,1753.90,ERRO DE LÓGICA
1,060cb4da-bec7-42f4-9f22-794213119d75,3479.10,1,1739.55,3479.10,ERRO DE LÓGICA
2,09c5b2b9-ae0c-4403-afb0-df81f5b2e7bb,3794.52,3,1897.26,11383.56,ERRO DE LÓGICA
3,0d061040-6477-4ba1-90e2-d4e81bead0e1,2558.15,4,1279.08,10232.60,ERRO DE LÓGICA
4,0ed109f8-cb73-4d26-a64b-3e8c12896a39,1393.66,1,696.83,1393.66,ERRO DE LÓGICA
5,183f58a0-955b-439b-a5e0-0f5405e8f906,2936.05,2,1468.03,5872.10,ERRO DE LÓGICA
6,200ac5f7-f334-41c7-aad8-a2573940bf7a,4439.06,4,2219.53,17756.24,ERRO DE LÓGICA
7,20fa9f30-6065-48a6-8891-056f6050f1c6,2588.15,4,1294.08,10352.60,ERRO DE LÓGICA
8,2510f73f-da23-4761-b180-0b24223c118e,3061.34,5,1530.67,15306.70,ERRO DE LÓGICA
9,26fce8ec-f07e-4097-b1b1-9d6a6224aac5,4237.39,4,2118.70,16949.56,ERRO DE LÓGICA


In [12]:
# Investigando Categorias Fora do Padrão

sql_investigacao_categoria = """
    SELECT 
        t.transaction_id,
        t.customer_name,
        
        -- A evidência do crime: O que está escrito no banco?
        t.category as categoria_encontrada,
        
        -- O que esperávamos (Contexto para o analista)
        'Eletrônicos, Livros, Casa, Moda, Brinquedos' as categorias_permitidas,
        
        'PADRONIZAÇÃO (Master Data)' as diagnostico
        
    FROM tb_vendas_typed t
    JOIN tb_data_quality_report r ON t.transaction_id = r.transaction_id
    WHERE r.erro_categoria_invalida = 1
    ORDER BY t.category -- Agrupa os erros iguais (ex: todos 'Automotivo' juntos)
    
"""

print("--- Transações com Categoria Desconhecida/Inválida ---")
df_categoria_errada = con.execute(sql_investigacao_categoria).df()

if df_categoria_errada.empty:
    print("⚠️ Nenhuma categoria inválida encontrada.")
    print("Dica: Se você não injetou 'lixo' (ex: categoria 'Jardinagem' ou 'Outros') no Python, isso é normal.")
else:
    display(df_categoria_errada)

--- Transações com Categoria Desconhecida/Inválida ---


,transaction_id,customer_name,categoria_encontrada,categorias_permitidas,diagnostico
0,bb811d6d-6502-4df1-a952-1765a5639c2d,Lorenzo Guerra,CASA,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
1,b1bfb477-1d4d-46c1-9419-3146da1e95f5,Hadassa Azevedo,CASA,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
2,2e9b5176-41d1-4897-ad89-d1270ef5196e,Maria Helena Carvalho,CASA,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
3,13499deb-3137-4dc6-803b-423e94445b86,Guilherme Lima,CASA,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
4,d49a375b-2c85-474e-8560-75940000e477,Maya Borges,CASA,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
...,...,...,...,...,...
173,c4e0ab9c-2e96-4ce2-a7b4-a83ee97e1b5c,João Felipe da Paz,eletronicos,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
174,3c80ea9a-0b31-4814-87b0-d4f140d3ca48,Anna Liz Pastor,eletronicos,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
175,746e75da-6491-45b4-83fc-2b0db020c88d,Asafe Mendes,eletronicos,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)
176,7316186e-1368-4cc5-873c-4c29e89019ad,Sr. Yuri Souza,eletronicos,"Eletrônicos, Livros, Casa, Moda, Brinquedos",PADRONIZAÇÃO (Master Data)


In [13]:
df_categoria_errada['categoria_encontrada'].unique()

array(['CASA', 'Casa', 'ELETRONICOS ', 'Eletro ', 'Eletrônicos', 'Home',
       'casa', 'eletronicos'], dtype=object)

## 🚀 Aplicação Prática: Onde isso entra no Pipeline de Dados?

Uma dúvida comum de Engenharia de Dados: *"Eu vou rodar isso na mão todo dia?"*
**Não.** Esses scripts de validação (SQL/DuckDB) são integrados no seu orquestrador de fluxo de dados (ex: **Airflow**, **Prefect**, **Dagster**).

Eles atuam como "portões de qualidade". Abaixo, três arquiteturas comuns de implementação:

### 🛑 Cenário A: O "Circuit Breaker" (Bloqueio Total)
*Ideal para dados críticos financeiros onde nenhum erro é tolerável.*

O pipeline roda a auditoria e verifica uma métrica de corte (Threshold).
1.  **Ingestão:** Dado bruto chega.
2.  **Auditoria:** O script roda `SELECT SUM(total_erros)`.
3.  **Decisão:**
    * Se `Erros > 5%` (ou qualquer regra definida):
    * 🔴 **PARA TUDO:** O Pipeline falha propositalmente (Exit Code 1).
    * 📢 **Alerta:** Envia notificação no Slack/Teams: *"Carga corrompida. Pipeline abortado para não sujar o Data Warehouse."*

### 😷 Cenário B: A Quarentena (Padrão Profissional)
*A abordagem mais robusta (Write-Audit-Publish). Garante que o BI nunca fique desatualizado, mas apenas com dados confiáveis.*

O fluxo se divide em dois caminhos após a auditoria:
1.  **Write:** Escreve os dados brutos.
2.  **Audit:** Classifica cada linha com 0 (Ok) ou 1 (Erro).
3.  **Publish (Divisão):**
    * ✅ **Caminho Feliz:** Linhas com `total_erros = 0` seguem para a camada **Silver/Gold** e alimentam dashboards.
    * ⚠️ **Caminho da Quarentena:** Linhas com `total_erros > 0` são desviadas para uma tabela de **Erros/Rejeitos**.
    * **Ação:** O Engenheiro analisa a tabela de erros no dia seguinte para corrigir a fonte, sem parar a empresa.



### 🔭 Cenário C: Observabilidade (Apenas Monitoramento)
*Focado em acompanhar a degradação do dado ao longo do tempo.*

O dado flui normalmente (mesmo com erros), mas a saúde é registrada.
1.  **Execução:** O dado é processado e entregue.
2.  **Histórico:** O resultado do nosso `sql_scorecard` é salvo em uma tabela de metadados (`tabela_historico_dq`).
3.  **Visualização:** O gerente acessa um dashboard de **Data Health**:
    * *"A qualidade dos dados de Vendas caiu de 99% para 85% nesta semana."*
    * Isso gera um ticket de manutenção, mas não bloqueia a operação imediata.

In [14]:
# ==============================================================================
# 🎯 ETAPA FINAL: O "SPLIT" (Arquitetura Write-Audit-Publish)
# Separando o joio do trigo baseados na auditoria anterior
# ==============================================================================

print("--- 🔄 INICIANDO ROTEAMENTO DE DADOS ---")

# 1. Definindo a Query dos DADOS BONS (Total Erros = 0)
# Apenas colunas de negócio, prontas para consumo.
sql_gold = """
    SELECT t.* FROM tb_vendas_typed t
    JOIN tb_data_quality_report r ON t.transaction_id = r.transaction_id
    WHERE r.total_erros = 0
"""

# 2. Definindo a Query da QUARENTENA (Total Erros > 0)
# Trazemos os dados originais + o diagnóstico do erro para facilitar a correção.
sql_quarantine = """
    SELECT 
        t.*, 
        r.erro_null_customer, r.erro_duplicidade, r.erro_calculo_matematico, 
        r.erro_preco_negativo, r.erro_data_futura, r.erro_categoria_invalida
    FROM tb_vendas_typed t
    JOIN tb_data_quality_report r ON t.transaction_id = r.transaction_id
    WHERE r.total_erros > 0
"""

# 3. Materializando os Dataframes (Extract)
df_gold = con.execute(sql_gold).df()
df_quarantine = con.execute(sql_quarantine).df()

# 4. Relatório de Carga (Load Report)
total_linhas = len(df_gold) + len(df_quarantine)
taxa_aprovacao = (len(df_gold) / total_linhas) * 100

print(f"📊 Processamento Concluído:")
print(f"   - Total Processado: {total_linhas} linhas")
print(f"   - ✅ Aprovados (Gold):     {len(df_gold)} linhas ({taxa_aprovacao:.1f}%)")
print(f"   - 🚫 Rejeitados (Quarentena): {len(df_quarantine)} linhas")
print("-" * 40)

# 5. Simulação de Publicação (Onde isso iria na vida real?)
if not df_gold.empty:
    print("🚀 [SUCESSO] df_gold enviado para Data Warehouse (Tabela: vendas_gold)")
    # display(df_gold.head(3)) # Opcional: mostrar uma amostra

if not df_quarantine.empty:
    print("⚠️ [ALERTA] df_quarantine enviado para S3 Bucket de Erros (Tabela: vendas_error_log)")
    print("   -> Ação Necessária: Time de Analytics deve corrigir a origem.")
    # Mostramos a quarentena para os alunos verem o resultado final dos erros
    display(df_quarantine.head(5))
else:
    print("✨ Parabéns! Carga 100% limpa. A quarentena está vazia.")

--- 🔄 INICIANDO ROTEAMENTO DE DADOS ---
📊 Processamento Concluído:
   - Total Processado: 2150 linhas
   - ✅ Aprovados (Gold):     1636 linhas (76.1%)
   - 🚫 Rejeitados (Quarentena): 514 linhas
----------------------------------------
🚀 [SUCESSO] df_gold enviado para Data Warehouse (Tabela: vendas_gold)
⚠️ [ALERTA] df_quarantine enviado para S3 Bucket de Erros (Tabela: vendas_error_log)
   -> Ação Necessária: Time de Analytics deve corrigir a origem.


,transaction_id,customer_id,customer_name,transaction_date_clean,unit_price_clean,quantity_clean,total_amount_clean,category,status,_raw_date,_raw_qty,erro_null_customer,erro_duplicidade,erro_calculo_matematico,erro_preco_negativo,erro_data_futura,erro_categoria_invalida
0,26ef9d11-367c-4c03-94c3-5d52d8e2463f,5506.0,Alice Viana,2099-12-31,3200.74,1,3200.74,Livros,Aprovado,2099-12-31,1,0,1,0,0,0,0
1,e3ed2e31-19f4-48a4-8902-a976a331f010,9935.0,Vitor Rezende,2099-12-31,706.29,1,706.29,Eletrônicos,Aprovado,2099-12-31,1,0,1,0,0,0,0
2,9b5976e0-3d20-4379-87ed-9b3bdd693ad3,4582.0,Srta. Alice da Rosa,2099-12-31,168.60,1,168.60,Livros,Aprovado,2099-12-31,1,0,1,0,0,0,0
3,666ae62d-1b21-497b-92e5-059f56b71d0e,7873.0,Enzo Fonseca,2099-12-31,2810.61,5,14053.05,Livros,Aprovado,2099-12-31,5,0,1,0,0,0,0
4,82bf8067-011c-4104-83e8-ae97c2900802,3615.0,Maya Mendes,2099-12-31,-2950.44,1,2950.44,Moda,Aprovado,2099-12-31,1,0,1,0,0,0,0
